# CASMI 2026 — CPU retrieval validation

Runs two held-out evaluations against training data: independent natural-product library spectra and a 0.5% metric-key structure holdout. The downloadable test file is deliberately not used for model selection.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

INPUT = Path('/kaggle/input')
competition_dirs = [p for p in INPUT.iterdir() if (p / 'train.parquet').exists() and (p / 'test.parquet').exists()]
source_files = list(INPUT.glob('**/prepare_validation.py'))
if not competition_dirs:
    raise FileNotFoundError('Attach the Enveda CASMI competition data input')
if not source_files:
    raise FileNotFoundError('Attach the versioned source-code Kaggle Dataset')
DATA_DIR = competition_dirs[0]
SOURCE_ROOT = source_files[0].parent
SRC = SOURCE_ROOT
sys.path.insert(0, str(SRC))
print('Data:', DATA_DIR)
print('Source:', SOURCE_ROOT)
commit_file = SOURCE_ROOT / 'SOURCE_COMMIT.txt'
print('Git commit:', commit_file.read_text().strip() if commit_file.exists() else 'missing provenance file')

# The score and duplicate handling must use the competition-pinned RDKit.
try:
    import rdkit
    parts = tuple(int(p) for p in rdkit.__version__.split('.')[:3])
except (ImportError, ValueError):
    parts = None
if parts != (2026, 3, 3):
    wheels = [p for p in INPUT.glob('**/*.whl') if 'rdkit' in p.name.lower() or '2026.03.3' in str(p)]
    if not wheels:
        raise RuntimeError('Attach the offline RDKit 2026.03.3 wheel dataset')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--upgrade', str(wheels[0])], check=True)
    import rdkit
    print('RDKit:', rdkit.__version__)


In [ ]:
import subprocess

TRAIN = str(DATA_DIR / 'train.parquet')
WORK = Path('/kaggle/working/validation')
WORK.mkdir(parents=True, exist_ok=True)

# Natural-product source holdout: references from other libraries can remain,
# which tests retrieval of known structures from independent spectra.
source_dir = WORK / 'natural_product_source'
subprocess.run([
    sys.executable, str(SRC / 'prepare_validation.py'), '--train', TRAIN,
    '--output-dir', str(source_dir), '--mode', 'source',
    '--query-source', 'enveda-np-examples',
], check=True)


In [ ]:
# Genuine structure holdout: all spectra for each selected metric-key are removed from reference.
structure_dir = WORK / 'structure_heldout'
subprocess.run([
    sys.executable, str(SRC / 'prepare_validation.py'), '--train', TRAIN,
    '--output-dir', str(structure_dir), '--mode', 'structure',
    '--fraction', '0.005', '--seed', '20260929',
], check=True)


In [ ]:
def score_split(name, folder):
    predictions = WORK / f'{name}_predictions.csv'
    subprocess.run([
        sys.executable, str(SRC / 'retrieve.py'),
        '--train', str(folder / 'reference.parquet'),
        '--test', str(folder / 'queries.parquet'),
        '--output', str(predictions), '--max-ppm', '30',
        '--fallback-ppm', '250', '--top-k', '25',
    ], check=True)
    subprocess.run([
        sys.executable, str(SRC / 'evaluate_predictions.py'),
        '--queries', str(folder / 'queries.parquet'),
        '--predictions', str(predictions),
    ], check=True)

score_split('natural_product_source', source_dir)
score_split('structure_heldout', structure_dir)


Save the JSON metrics and notebook version with the Git source commit. Keep this as the baseline before testing rankers or GPU models.
